# Task A 替换候选校准（cond_wordproblem_v1，整数，仅加减）
运行时：GPU。算力 < 8（T4）使用 half、显存 0.80、上下文 8192、enforce_eager；更好的 GPU 使用 bfloat16。依次运行全部单元，最后下载 raw_outputs.jsonl 和 env.json，放到 `Experiments/task_a_wordproblem/calib_int/` 后运行 analyze.py。
本单元内嵌的是加减题，没有乘法。校准结果尚未产出。


In [1]:
!pip install -q vllm
!nvidia-smi

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
cudf-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
numba-cuda 0.28.2 requires cuda-core<1.0.0,>=0.5.1, but you have cuda-core 1.3.0 which is incompatible.
cuml-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
gradio 6.31.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
Wed Nov 11 18:05:59 2026       
+------------------------------------------------------------

In [2]:
import base64,gzip
open('calib_slim.jsonl','wb').write(gzip.decompress(base64.b64decode('...')))
open('run_calib.py','w',encoding='utf-8').write(base64.b64decode('IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiLmlbTmlbDliqDlh4/niYjmnaHku7blupTnlKjpopjnmoTmoKHlh4bmjqjnkIbvvIhjb25kX3dvcmRwcm9ibGVtX3Yx77yJ44CCCgrlnKjmnIkgR1BVIOeahOacuuWZqOS4iui/kOihjO+8iOS+i+WmgiBDb2xhYu+8ieOAguacrOacuuayoeaciSBDVURBIOaXtuS4jeimgeaJp+ihjOOAggrovpPlhaXvvJpjYWxpYl9zbGltLmpzb25s77yIcWlkLCBkLCBpLCBxLCBhbnMsIGFuc3dlcl9kZWNpbWFsc++8iQogICAgICBzZWVkIDEyMzTvvIzkuLLooYwgZCDiiIggezEsMiwzLDUsMTAsMTUsMjB9IMOXIDEwMArovpPlh7rvvJpyYXdfb3V0cHV0cy5qc29ubO+8iOavj+mimCDDlyBEQS9Db1Qg5LiA6KGM77yJ77yMZW52Lmpzb24KCumHh+agt+S4jiBFeHBlcmltZW50cy90YXNrX2Eg5LiA6Ie077yadGVtcGVyYXR1cmUgMC4244CBdG9wX3AgMC45NeOAggp0b3Bfaz0yMCDlj6rorrDlvZXvvIzkuI3kvKDlhaUgU2FtcGxpbmdQYXJhbXPjgIIKUXdlbjMtNEItSW5zdHJ1Y3QtMjUwNyDkuI3mmK8gdGhpbmtpbmcg5qih5Z6L77yM5LiN5LygIGVuYWJsZV90aGlua2luZ+OAggpEQSAvIENvVCDnmoQgbWF4X3Rva2VucyDmjInmnKzmoKHlh4bnmoTovpPlh7rplb/luqborr7lrprvvIhEQSA2NO+8jENvVCAyMDQ477yJ77yMCuS4jeaYr+WFqOefqemYteWunumqjOeahCA2MTQ044CCCiIiIgppbXBvcnQgaGFzaGxpYgppbXBvcnQgaW1wb3J0bGliLm1ldGFkYXRhIGFzIG1kCmltcG9ydCBqc29uCmltcG9ydCBvcwppbXBvcnQgcmUKaW1wb3J0IHN5cwppbXBvcnQgdGltZQoKIyDlv4XpobvlnKggaW1wb3J0IHZsbG0g5LmL5YmN6K6+572u44CC5YWz6ZetIEZsYXNoSW5mZXIg6YeH5qC35Zmo77yM6YG/5YWN5rKh5pyJIG52Y2Mg5pe2IEpJVCDlpLHotKXjgIIKb3MuZW52aXJvblsiVkxMTV9VU0VfRkxBU0hJTkZFUl9TQU1QTEVSIl0gPSAiMCIKb3MuZW52aXJvblsiVkxMTV9FTkFCTEVfVjFfTVVMVElQUk9DRVNTSU5HIl0gPSAiMCIKb3MuZW52aXJvblsiVkxMTV9MT0dHSU5HX0xFVkVMIl0gPSAiREVCVUciCgpNT0RFTCA9IG9zLmVudmlyb24uZ2V0KCJNT0RFTCIsICJRd2VuL1F3ZW4zLTRCLUluc3RydWN0LTI1MDciKQpSRVFVRVNURURfUkVWSVNJT04gPSBvcy5lbnZpcm9uLmdldCgiTU9ERUxfUkVWSVNJT04iKSBvciBOb25lCklOID0gc3lzLmFyZ3ZbMV0gaWYgbGVuKHN5cy5hcmd2KSA+IDEgZWxzZSAiY2FsaWJfc2xpbS5qc29ubCIKT1VUID0gc3lzLmFyZ3ZbMl0gaWYgbGVuKHN5cy5hcmd2KSA+IDIgZWxzZSAicmF3X291dHB1dHMuanNvbmwiCiMgZCDlpKfkuo7or6XlgLzml7bvvIxDb1Qg5Y+q6LeR5YmNIENPVF9OX0hJR0gg5p2h77yIaSDku44gMCDotbfvvInjgILpu5jorqQgMjDvvIzljbMgNzAwIOmimOWFqOmDqOi3kSBDb1TjgIIKQ09UX01BWF9EID0gaW50KG9zLmVudmlyb24uZ2V0KCJDT1RfRlVMTF9VUF9UTyIsICIyMCIpKQpDT1RfTl9ISUdIID0gaW50KG9zLmVudmlyb24uZ2V0KCJDT1RfTl9ISUdIIiwgIjEwMCIpKQpQQVJBTVMgPSB7InRlbXBlcmF0dXJlIjogMC42LCAidG9wX3AiOiAwLjk1LCAidG9wX2tfcmVjb3JkZWRfbm90X3Bhc3NlZCI6IDIwfQpNQVhfVE9LRU5TID0geyJEQSI6IDY0LCAiQ29UIjogMjA0OH0KU1lTVEVNID0gewogICAgIkRBIjogKCJTb2x2ZSBzaWxlbnRseS4gRG8gbm90IHNob3cgY2FsY3VsYXRpb25zLCBpbnRlcm1lZGlhdGUgdmFsdWVzLCBleHBsYW5hdGlvbnMgb3IgY29tbWVudHMuICIKICAgICAgICAgICAiT3V0cHV0IG9ubHkgdGhlIGZpbmFsIGFuc3dlciBsaW5lLlxuIgogICAgICAgICAgICJUaGUgZmluYWwgbGluZSBtdXN0IGJlICMjIyMgZm9sbG93ZWQgYnkgdGhlIGZpbmFsIG51bWJlciwgd2l0aCBub3RoaW5nIGFmdGVyIGl0LiIpLAogICAgIkNvVCI6ICgiU29sdmUgc3RlcCBieSBzdGVwLiBFeHBsaWNpdGx5IHdyaXRlIHRoZSBhbW91bnQgYWZ0ZXIgZXZlcnkgZXZlbnQsIG9uZSBldmVudCBwZXIgbGluZS4gIgogICAgICAgICAgICAiVGhlbiBnaXZlIHRoZSBmaW5hbCBhbnN3ZXIuXG4iCiAgICAgICAgICAgICJUaGUgZmluYWwgbGluZSBtdXN0IGJlICMjIyMgZm9sbG93ZWQgYnkgdGhlIGZpbmFsIG51bWJlciwgd2l0aCBub3RoaW5nIGFmdGVyIGl0LiIpLAp9CgoKZGVmIHNlZWRfb2YocWlkLCBtb2RlKToKICAgIHJldHVybiBpbnQoaGFzaGxpYi5zaGEyNTYoZiJ7cWlkfXx7bW9kZX0iLmVuY29kZSgpKS5oZXhkaWdlc3QoKVs6OF0sIDE2KQoKCmRlZiByZXNvbHZlX21vZGVsX3JldmlzaW9uKG1vZGVsLCByZXF1ZXN0ZWQpOgogICAgIiIi5Zyo5Yib5bu6IExMTSDkuYvliY3miorov5znq68gbWFpbi90YWcg6Kej5p6Q5oiQIGNvbW1pdOOAguS4jeivuyB2TExNIOeahOengeaciSBjb21taXQg5a2X5q6144CCIiIiCiAgICBmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCiAgICBsb2NhbCA9IFBhdGgobW9kZWwpCiAgICBpZiBsb2NhbC5pc19kaXIoKToKICAgICAgICBzb3VyY2UgPSBsb2NhbC5yZXNvbHZlKCkKICAgICAgICBpZiBzb3VyY2UucGFyZW50Lm5hbWUgPT0gInNuYXBzaG90cyIgYW5kIHJlLmZ1bGxtYXRjaChyIlswLTlhLWZBLUZdezQwfSIsIHNvdXJjZS5uYW1lKToKICAgICAgICAgICAgY29tbWl0ID0gc291cmNlLm5hbWUubG93ZXIoKQogICAgICAgICAgICBpZiByZXF1ZXN0ZWQgaXMgbm90IE5vbmUgYW5kIHJlcXVlc3RlZC5sb3dlcigpICE9IGNvbW1pdDoKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIuacrOWcsCBzbmFwc2hvdCDkuI4gTU9ERUxfUkVWSVNJT04g5LiN5LiA6Ie044CCIikKICAgICAgICAgICAgcmV0dXJuIHsic291cmNlIjogc3RyKHNvdXJjZSksICJtb2RlbF9jb21taXQiOiBjb21taXQsCiAgICAgICAgICAgICAgICAgICAgInJlcXVlc3RlZF9tb2RlbF9yZXZpc2lvbiI6IHJlcXVlc3RlZCwgInJldmlzaW9uX2tpbmQiOiAibG9jYWxfc25hcHNob3QiLAogICAgICAgICAgICAgICAgICAgICJsb2FkaW5nX2t3YXJncyI6IHt9fQogICAgICAgIGlmIHJlcXVlc3RlZCBpcyBOb25lOgogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIuaZrumAmuacrOWcsOadg+mHjeebruW9leayoeacieWPr+mqjOivgeeahCBjb21taXTvvJvor7forr7nva4gTU9ERUxfUkVWSVNJT07jgIIiKQogICAgICAgIHJldHVybiB7InNvdXJjZSI6IHN0cihzb3VyY2UpLCAibW9kZWxfY29tbWl0IjogTm9uZSwKICAgICAgICAgICAgICAgICJyZXF1ZXN0ZWRfbW9kZWxfcmV2aXNpb24iOiByZXF1ZXN0ZWQsICJyZXZpc2lvbl9raW5kIjogImxvY2FsX2xhYmVsIiwKICAgICAgICAgICAgICAgICJsb2FkaW5nX2t3YXJncyI6IHt9fQoKICAgIGlmIHJlcXVlc3RlZCBpcyBub3QgTm9uZSBhbmQgcmUuZnVsbG1hdGNoKHIiWzAtOWEtZkEtRl17NDB9IiwgcmVxdWVzdGVkKToKICAgICAgICBjb21taXQgPSByZXF1ZXN0ZWQubG93ZXIoKQogICAgZWxzZToKICAgICAgICBmcm9tIGh1Z2dpbmdmYWNlX2h1YiBpbXBvcnQgSGZBcGkKICAgICAgICBpbmZvID0gSGZBcGkoKS5tb2RlbF9pbmZvKHJlcG9faWQ9bW9kZWwsIHJldmlzaW9uPXJlcXVlc3RlZCBvciAibWFpbiIpCiAgICAgICAgY29tbWl0ID0gaW5mby5zaGEKICAgICAgICBpZiBub3QgaXNpbnN0YW5jZShjb21taXQsIHN0cikgb3IgcmUuZnVsbG1hdGNoKHIiWzAtOWEtZkEtRl17NDB9IiwgY29tbWl0KSBpcyBOb25lOgogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIkh1Z2dpbmcgRmFjZSDmnKrov5Tlm57mnInmlYggY29tbWl044CC5Y+v6K6+572uIE1PREVMX1JFVklTSU9OIOS4uuWujOaVtCBjb21taXTjgIIiKQogICAgICAgIGNvbW1pdCA9IGNvbW1pdC5sb3dlcigpCiAgICByZXR1cm4geyJzb3VyY2UiOiBtb2RlbCwgIm1vZGVsX2NvbW1pdCI6IGNvbW1pdCwKICAgICAgICAgICAgInJlcXVlc3RlZF9tb2RlbF9yZXZpc2lvbiI6IHJlcXVlc3RlZCwgInJldmlzaW9uX2tpbmQiOiAiaHViX2NvbW1pdCIsCiAgICAgICAgICAgICJsb2FkaW5nX2t3YXJncyI6IHsicmV2aXNpb24iOiBjb21taXQsICJ0b2tlbml6ZXJfcmV2aXNpb24iOiBjb21taXR9fQoKCmRlZiBsb2FkaW5nX2t3YXJncyh0b3JjaCwgc291cmNlLCByZXZpc2lvbik6CiAgICAiIiJUNO+8iOeul+WKmyA8IDjvvInmsr/nlKggQUdFTlRTLm1kIDMuNO+8m+abtOWlveeahCBHUFUg55SoIGJmMTbvvIzlubbkv53nlZnmnKzmoKHlh4bljp/mnaXnmoTmmL7lrZjplb/luqbjgIIiIiIKICAgIGNhcCA9IHRvcmNoLmN1ZGEuZ2V0X2RldmljZV9jYXBhYmlsaXR5KDApCiAgICB2aXNpYmxlX2dpYiA9IHRvcmNoLmN1ZGEuZ2V0X2RldmljZV9wcm9wZXJ0aWVzKDApLnRvdGFsX21lbW9yeSAvIDIqKjMwCiAgICBpZiBjYXBbMF0gPCA4OgogICAgICAgIGlmIHZpc2libGVfZ2liIDwgMTQ6CiAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiVHVyaW5nIC8gVDQg5LiK55qEIDRCIOiHs+WwkemcgOimgSAxNCBHaUIg5Y+v6KeB5pi+5a2Y44CCIikKICAgICAgICBkdHlwZSA9ICJoYWxmIgogICAgICAgIGxvYWRpbmcgPSBkaWN0KG1vZGVsPXNvdXJjZSwgZHR5cGU9ImhhbGYiLCBncHVfbWVtb3J5X3V0aWxpemF0aW9uPTAuODAsCiAgICAgICAgICAgICAgICAgICAgICAgbWF4X21vZGVsX2xlbj04MTkyLCBtYXhfbnVtX3NlcXM9MSwgZW5mb3JjZV9lYWdlcj1UcnVlLCBzZWVkPTApCiAgICBlbHNlOgogICAgICAgIGR0eXBlID0gImJmbG9hdDE2IgogICAgICAgIGxvYWRpbmcgPSBkaWN0KG1vZGVsPXNvdXJjZSwgZHR5cGU9ImJmbG9hdDE2IiwgZ3B1X21lbW9yeV91dGlsaXphdGlvbj0wLjg4LAogICAgICAgICAgICAgICAgICAgICAgIG1heF9tb2RlbF9sZW49NDA5Niwgc2VlZD0wKQogICAgbG9hZGluZy51cGRhdGUocmV2aXNpb25bImxvYWRpbmdfa3dhcmdzIl0pCiAgICByZXR1cm4gZHR5cGUsIGxpc3QoY2FwKSwgbG9hZGluZwoKCmRlZiBtYWluKCk6CiAgICBpbXBvcnQgdG9yY2gKICAgIGlmIG5vdCB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigi5pys5py65rKh5pyJIENVREHjgILor7flnKggQ29sYWIgR1BVIOS4iui/kOihjO+8jOS4jeimgeaKiue8uiBHUFUg5b2T5oiQ5Luj56CB6ZSZ6K+v44CCIikKICAgIHJldmlzaW9uID0gcmVzb2x2ZV9tb2RlbF9yZXZpc2lvbihNT0RFTCwgUkVRVUVTVEVEX1JFVklTSU9OKQogICAgZHR5cGUsIGNhcCwgbG9hZGluZyA9IGxvYWRpbmdfa3dhcmdzKHRvcmNoLCByZXZpc2lvblsic291cmNlIl0sIHJldmlzaW9uKQogICAgZnJvbSB2bGxtIGltcG9ydCBMTE0sIFNhbXBsaW5nUGFyYW1zCgogICAgcm93cyA9IFtqc29uLmxvYWRzKGxpbmUpIGZvciBsaW5lIGluIG9wZW4oSU4pXQogICAgbGxtID0gTExNKCoqbG9hZGluZykKICAgIHRvayA9IGxsbS5nZXRfdG9rZW5pemVyKCkKICAgIGpvYnMgPSBbXQogICAgZm9yIHJvdyBpbiByb3dzOgogICAgICAgIGZvciBtb2RlIGluICgiREEiLCAiQ29UIik6CiAgICAgICAgICAgIGlmIG1vZGUgPT0gIkNvVCIgYW5kIHJvd1siZCJdID4gQ09UX01BWF9EIGFuZCByb3dbImkiXSA+PSBDT1RfTl9ISUdIOgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgam9icy5hcHBlbmQoKHJvdywgbW9kZSkpCiAgICBwcm9tcHRzID0gW3Rvay5hcHBseV9jaGF0X3RlbXBsYXRlKAogICAgICAgIFt7InJvbGUiOiAic3lzdGVtIiwgImNvbnRlbnQiOiBTWVNURU1bbW9kZV19LCB7InJvbGUiOiAidXNlciIsICJjb250ZW50Ijogcm93WyJxIl19XSwKICAgICAgICB0b2tlbml6ZT1GYWxzZSwgYWRkX2dlbmVyYXRpb25fcHJvbXB0PVRydWUpIGZvciByb3csIG1vZGUgaW4gam9ic10KICAgIHBhcmFtcyA9IFtTYW1wbGluZ1BhcmFtcyh0ZW1wZXJhdHVyZT1QQVJBTVNbInRlbXBlcmF0dXJlIl0sIHRvcF9wPVBBUkFNU1sidG9wX3AiXSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICBtYXhfdG9rZW5zPU1BWF9UT0tFTlNbbW9kZV0sIHNlZWQ9c2VlZF9vZihyb3dbInFpZCJdLCBtb2RlKSkKICAgICAgICAgICAgICBmb3Igcm93LCBtb2RlIGluIGpvYnNdCiAgICB0MCA9IHRpbWUudGltZSgpCiAgICBvdXRzID0gbGxtLmdlbmVyYXRlKHByb21wdHMsIHBhcmFtcykKICAgIGVsYXBzZWQgPSB0aW1lLnRpbWUoKSAtIHQwCiAgICB3aXRoIG9wZW4oT1VULCAidyIpIGFzIGY6CiAgICAgICAgZm9yIChyb3csIG1vZGUpLCBvdXQgaW4gemlwKGpvYnMsIG91dHMpOgogICAgICAgICAgICBjaG9pY2UgPSBvdXQub3V0cHV0c1swXQogICAgICAgICAgICBmLndyaXRlKGpzb24uZHVtcHMoewogICAgICAgICAgICAgICAgInFpZCI6IHJvd1sicWlkIl0sICJkIjogcm93WyJkIl0sICJpIjogcm93WyJpIl0sICJtb2RlIjogbW9kZSwgImdvbGQiOiByb3dbImFucyJdLAogICAgICAgICAgICAgICAgImFuc3dlcl9kZWNpbWFscyI6IHJvd1siYW5zd2VyX2RlY2ltYWxzIl0sICJ0ZXh0IjogY2hvaWNlLnRleHQsCiAgICAgICAgICAgICAgICAiZmluaXNoX3JlYXNvbiI6IGNob2ljZS5maW5pc2hfcmVhc29uLCAibl9vdXRfdG9rZW5zIjogbGVuKGNob2ljZS50b2tlbl9pZHMpLAogICAgICAgICAgICAgICAgIm5fcHJvbXB0X3Rva2VucyI6IGxlbihvdXQucHJvbXB0X3Rva2VuX2lkcyksICJzZWVkIjogc2VlZF9vZihyb3dbInFpZCJdLCBtb2RlKSwKICAgICAgICAgICAgfSkgKyAiXG4iKQogICAgZW52ID0gewogICAgICAgICJtb2RlbCI6IE1PREVMLAogICAgICAgICJtb2RlbF9jb21taXQiOiByZXZpc2lvblsibW9kZWxfY29tbWl0Il0sCiAgICAgICAgInJlcXVlc3RlZF9tb2RlbF9yZXZpc2lvbiI6IHJldmlzaW9uWyJyZXF1ZXN0ZWRfbW9kZWxfcmV2aXNpb24iXSwKICAgICAgICAicmV2aXNpb25fa2luZCI6IHJldmlzaW9uWyJyZXZpc2lvbl9raW5kIl0sCiAgICAgICAgImR0eXBlIjogZHR5cGUsCiAgICAgICAgImdwdSI6IHRvcmNoLmN1ZGEuZ2V0X2RldmljZV9uYW1lKDApLAogICAgICAgICJjb21wdXRlX2NhcGFiaWxpdHkiOiBjYXAsCiAgICAgICAgImxvYWRpbmciOiB7azogbG9hZGluZ1trXSBmb3IgayBpbiBsb2FkaW5nIGlmIGsgIT0gIm1vZGVsIn0sCiAgICAgICAgIm1vZGVsX3NvdXJjZSI6IGxvYWRpbmdbIm1vZGVsIl0sCiAgICAgICAgInZsbG0iOiBtZC52ZXJzaW9uKCJ2bGxtIiksCiAgICAgICAgInRvcmNoIjogdG9yY2guX192ZXJzaW9uX18sCiAgICAgICAgInRyYW5zZm9ybWVycyI6IG1kLnZlcnNpb24oInRyYW5zZm9ybWVycyIpLAogICAgICAgICJzYW1wbGluZyI6IFBBUkFNUywKICAgICAgICAibWF4X3Rva2VucyI6IE1BWF9UT0tFTlMsCiAgICAgICAgInN5c3RlbV9wcm9tcHRzIjogU1lTVEVNLAogICAgICAgICJjaGF0X3RlbXBsYXRlX2t3YXJncyI6IHt9LAogICAgICAgICJuX2pvYnMiOiBsZW4oam9icyksCiAgICAgICAgImdlbmVyYXRlX3NlY29uZHMiOiByb3VuZChlbGFwc2VkLCAxKSwKICAgICAgICAiY290X2Z1bGxfdXBfdG9fZCI6IENPVF9NQVhfRCwKICAgICAgICAiY290X25faGlnaCI6IENPVF9OX0hJR0gsCiAgICAgICAgInZsbG1fZW52Ijoge2s6IG9zLmVudmlyb24uZ2V0KGspIGZvciBrIGluICgKICAgICAgICAgICAgIlZMTE1fVVNFX0ZMQVNISU5GRVJfU0FNUExFUiIsICJWTExNX0VOQUJMRV9WMV9NVUxUSVBST0NFU1NJTkciLCAiVkxMTV9MT0dHSU5HX0xFVkVMIil9LAogICAgfQogICAganNvbi5kdW1wKGVudiwgb3BlbigiZW52Lmpzb24iLCAidyIpLCBpbmRlbnQ9MikKICAgIHByaW50KGpzb24uZHVtcHMoZW52LCBpbmRlbnQ9MikpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo=').decode())
print('files written')


files written


In [3]:
!python run_calib.py calib_slim.jsonl raw_outputs.jsonl

W1111 18:06:11.098000 2033 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1111 18:06:11.099000 2033 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1111 18:06:11.099000 2033 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead
(MainProcess pid=2033) INFO 11-11 18:06:22 [entrypoints/common/startup.py:96] non-default args: {'dtype': 'bfloat16', 'max_model_len': 4096, 'gpu_memory_utilization': 0.88, 'disable_log_stats': True, 'revision': 'cdbee75f17c01a7cc42f958dc650907174af0554', 'tokenizer_revision': 'cdbee75f17c01a7cc42f958dc650907174af0554', 'model': 'Qwen/Qwen3-4B-Instruct-2507'}
config.json: 100% 727/727 [00:00<00:00, 3.40MB/s]
(MainProcess pid=2033) WARNING 11-11 18:06:22 [engine/arg_utils.py:1984] The global random seed is set to 0. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python pro

In [4]:
from google.colab import files
files.download('raw_outputs.jsonl'); files.download('env.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>